# ARCHS4 model building QC

Sanity checks on the ARCHS4 CLAMP models: matrix shape after filtering, the
inferred rank, and how much of the pathway prior each seed actually recovers.

The heavy work happens in `scripts/archs4/` under `workflow/rules/archs4.smk`;
this notebook only reads the small summary tables those rules leave behind. It
writes CSVs and saves no figures - publication panels live in `nbs/99_panels/`.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
    library(data.table)
    library(ggplot2)
    library(yaml)
    library(here)
})

cfg <- yaml::read_yaml(here("config.yaml"))
MODEL_COLORS <- unlist(cfg$MODEL_COLORS)

## Settings

In [ ]:
SEEDS    <- as.integer(snakemake@params[["seeds"]])
FDR_CUT  <- as.numeric(snakemake@params[["fdr"]])
OUT_DIR  <- here(snakemake@params[["out_dir"]])
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

summary_files <- unlist(snakemake@input[["summaries"]])
names(summary_files) <- paste0("seed", SEEDS)

meta <- readRDS(snakemake@input[["metadata"]])
stopifnot(length(summary_files) == length(SEEDS))

## Filtered expression matrix

In [ ]:
matrix_qc <- data.table(
    n_genes   = meta$n_genes_thin,
    n_samples = meta$n_samples
)
stopifnot(matrix_qc$n_genes > 0, matrix_qc$n_samples > 0)
matrix_qc

## Pathway recovery per seed

`summary.csv` holds one row per (pathway, LV) association CLAMP kept. Two things
matter for QC: how many distinct gene sets the model recovers at the FDR
threshold, and whether the three seeds agree. A seed that recovers far fewer
pathways than its siblings usually means the fit did not converge.

In [ ]:
per_seed <- rbindlist(lapply(names(summary_files), function(nm) {
    s <- fread(summary_files[[nm]])
    stopifnot(all(c("pathway", "LV", "AUC", "FDR") %in% names(s)))
    sig <- s[FDR < FDR_CUT]
    data.table(
        seed              = nm,
        n_associations    = nrow(s),
        n_sig             = nrow(sig),
        n_pathways_sig    = uniqueN(sig$pathway),
        n_lvs_sig         = uniqueN(sig$LV),
        n_lvs_total       = uniqueN(s$LV),
        median_auc_sig    = if (nrow(sig)) median(sig$AUC) else NA_real_
    )
}))
per_seed[, frac_lvs_sig := n_lvs_sig / n_lvs_total]
per_seed

In [ ]:
ggplot(per_seed, aes(x = seed, y = n_pathways_sig)) +
    geom_col(fill = MODEL_COLORS[["CLAMPfull"]], width = 0.6) +
    geom_text(aes(label = n_pathways_sig), vjust = -0.4, size = 3) +
    labs(
        x = NULL,
        y = sprintf("Distinct gene sets recovered (FDR < %s)", FDR_CUT),
        title = "ARCHS4 CLAMPfull pathway recovery by seed"
    ) +
    theme_classic(base_size = 11) +
    theme(panel.grid = element_blank())

## Seed agreement

Jaccard overlap of the recovered gene-set collections between every pair of
seeds. Low overlap would mean the model is not identifiable at this rank.

In [ ]:
sig_sets <- lapply(summary_files, function(f) {
    s <- fread(f, select = c("pathway", "FDR"))
    unique(s[FDR < FDR_CUT]$pathway)
})

pairs <- combn(names(sig_sets), 2, simplify = FALSE)
overlap <- rbindlist(lapply(pairs, function(p) {
    a <- sig_sets[[p[1]]]; b <- sig_sets[[p[2]]]
    data.table(
        seed_a = p[1], seed_b = p[2],
        n_a = length(a), n_b = length(b),
        n_shared = length(intersect(a, b)),
        jaccard = length(intersect(a, b)) / length(union(a, b))
    )
}))
overlap

## Write QC tables

In [ ]:
fwrite(matrix_qc, snakemake@output[["matrix_qc"]])
fwrite(per_seed,  snakemake@output[["seed_qc"]])
fwrite(overlap,   snakemake@output[["seed_overlap"]])
cat("Wrote QC tables to", OUT_DIR, "\n")